In [2]:
import sys
import pandas as pd
sys.path.append('../Source')

from Pipeline.clean_ACLED import cleanACLED
from Pipeline.clean_market import cleanMarket

acledData = cleanACLED('../Data/raw/ACLED Data_2026-06-24.csv')
marketData = cleanMarket('../data/raw/rawMarket.csv')

print(f'ACLED: {acledData.shape[0]:,} rows')
print(f'Market: {marketData.shape[0]:,} trading days')
print(f'Market date range: {marketData.index.min()} to {marketData.index.max()}')


ACLED: 1,087,171 rows
Market: 2,129 trading days
Market date range: 2018-01-02 00:00:00 to 2026-06-23 00:00:00


In [3]:
tradingDays = pd.DatetimeIndex(marketData.index)

print(f'Total trading days: {len(tradingDays):,}')
print(f'First trading day: {tradingDays[0]}')
print(f'Last trading day: {tradingDays[-1]}')

weekends = tradingDays[tradingDays.weekday >= 5]
print(f'Weekend days in trading calendar: {len(weekends)}')

Total trading days: 2,129
First trading day: 2018-01-02 00:00:00
Last trading day: 2026-06-23 00:00:00
Weekend days in trading calendar: 0


In [9]:
def matchToTradingDay(eventDate, tradingDays): 
    """
    Match an event date to the nearest trading day.
    
    Arguments:
        eventDate: The date of the event.
    tradingDays: The index of valid trading days.
    
    Returns:
        The nearest trading day 
    
    """
    index = tradingDays.searchsorted(eventDate)
    index = min(index, len(tradingDays) - 1)
    return tradingDays[index]

testDates = ['2022-02-26', '2022-02-27', '2022-02-28']
for date in testDates: 
    matchedDate = matchToTradingDay(pd.Timestamp(date), tradingDays)
    print(f'{date} {pd.Timestamp(date).day_name()} -> {matchedDate.date()}')

2022-02-26 Saturday -> 2022-02-28
2022-02-27 Sunday -> 2022-02-28
2022-02-28 Monday -> 2022-02-28


In [11]:
acledData['tradingDay'] = acledData['event_date'].apply(
    lambda x: matchToTradingDay(x, tradingDays)
)

print('tradingDay column added to ACLED data')
print(f'Sample of matched days: ')
acledData[['event_date', 'tradingDay']].head(10)

tradingDay column added to ACLED data
Sample of matched days: 


,event_date,tradingDay
0,2018-01-01,2018-01-02
1,2018-01-01,2018-01-02
2,2018-01-01,2018-01-02
3,2018-01-01,2018-01-02
4,2018-01-01,2018-01-02
5,2018-01-01,2018-01-02
6,2018-01-01,2018-01-02
7,2018-01-01,2018-01-02
8,2018-01-01,2018-01-02
9,2018-01-01,2018-01-02
